# Notebook 04: Component 1 – Multi-Class Product Classification & Temporal Drift

**ComplaintScope: Consumer-Complaint Triage and Early Warning System**  
**Course:** CBS3009 – Advanced Social, Text and Media Analytics  
**Authors:** Ruhaan bansal (23BBS0046), Vedant Rajesh (23BBS0037), Anish Desai (23BBS0119)  
**Faculty:** Sathish Kumar K | **Institution:** VIT Vellore

### Objectives:
1. Train and evaluate an 8-model classification benchmark suite on chronological partitions.
2. Conduct chronological hyperparameter optimization on validation split (logged to `results/tables/classification_grid.csv`).
3. Evaluate final performance on unseen chronological test split with 95% bootstrap confidence intervals.
4. Perform pairwise McNemar's statistical significance tests comparing tuned models against baselines.
5. Quantify temporal concept drift (AQ3 experiment: Chronological vs. Random split performance).
6. Conduct qualitative error taxonomy analysis across confusion pairs.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.config import load_config, get_path
print('Setup complete. All classification modules ready.')

## 1. Multi-Model Benchmark Suite

We evaluate 8 distinct model families spanning heuristic baselines, naive Bayes, decision trees, calibrated linear SVMs, tuned logistic regression, and dense neural embeddings (`all-MiniLM-L6-v2` + Logistic Regression).

In [ ]:
results_path = Path('../results/tables/classification_results.csv') if Path('../results/tables/classification_results.csv').exists() else Path('results/tables/classification_results.csv')
if results_path.exists():
    df_results = pd.read_csv(results_path)
    print('Classification Performance Comparison (Chronological Test Split):')
    display_cols = ['model', 'val_macro_f1', 'test_macro_f1', 'test_accuracy', 'ci_lower', 'ci_upper']
    print(df_results[[c for c in display_cols if c in df_results.columns]].to_string(index=False))
else:
    print('Classification results table not found yet at', results_path)

## 2. Chronological Hyperparameter Tuning

Hyperparameter sweeps for regularization strength $C$ and optimization solvers were evaluated strictly on the validation partition to avoid test contamination.

In [ ]:
grid_path = Path('../results/tables/classification_grid.csv') if Path('../results/tables/classification_grid.csv').exists() else Path('results/tables/classification_grid.csv')
if grid_path.exists():
    df_grid = pd.read_csv(grid_path)
    print('Hyperparameter Grid Search Results (Validation Split):')
    print(df_grid.to_string(index=False))
else:
    print('Grid search table not found yet.')

## 3. Confusion Matrix & Per-Class Analysis

Analyzing misclassification patterns highlights boundary overlap between related product categories (e.g., Credit Cards vs. Checking Accounts).

In [ ]:
cm_img = Path('../results/figures/fig10_confusion_matrix.png') if Path('../results/figures/fig10_confusion_matrix.png').exists() else Path('results/figures/fig10_confusion_matrix.png')
if cm_img.exists():
    from IPython.display import Image, display
    display(Image(filename=str(cm_img), width=850))
else:
    print('Confusion matrix figure not found at', cm_img)

## 4. Analytical Question 3 (AQ3): Temporal Concept Drift Quantification

We compare test Macro-F1 under **Chronological Split** (training on past, testing on future) versus **Random Split** (random 70/15/15 sampling across all years). The performance gap quantifies temporal vocabulary and product distribution drift.

In [ ]:
drift_path = Path('../results/tables/aq3_drift_results.csv') if Path('../results/tables/aq3_drift_results.csv').exists() else Path('results/tables/aq3_drift_results.csv')
if drift_path.exists():
    df_drift = pd.read_csv(drift_path)
    print('AQ3 Temporal Drift Evaluation:')
    print(df_drift.to_string(index=False))
else:
    print('AQ3 drift table not found yet.')

## 5. Qualitative Error Taxonomy Audit

We manually categorize 60 misclassified narratives into a formal error taxonomy:
1. **Multi-Product Overlap:** Complaints involving both a checking account overdraft and a linked credit line.
2. **Ambiguous Terminology:** General payment dispute terms shared across debt collection and loan servicing.
3. **Reporting Bias:** Credit bureau reporting issues stemming from disputed personal loans.

In [ ]:
error_path = Path('../results/tables/classification_error_taxonomy.csv') if Path('../results/tables/classification_error_taxonomy.csv').exists() else Path('results/tables/classification_error_taxonomy.csv')
if error_path.exists():
    df_err = pd.read_csv(error_path)
    print(f'Audited Error Taxonomy Samples ({len(df_err)} cases):')
    print(df_err.head(10).to_string(index=False))
else:
    print('Error taxonomy table not found yet.')